Train, evaluate, and fine-tune a logistic regression model for multi-class classification problems, incorporating regularization techniques and performing model interpretability analysis with coefficients and odds ratios.

In [1]:
import pandas as pd
import numpy as np

In [2]:
np.random.seed(42)

In [3]:
n_samples = 400

In [4]:
age = np.random.normal(48, 15, n_samples).clip(18, 85)
refill_frequency = np.random.poisson(lam=4, size=n_samples)
monthly_spend = np.random.normal(120, 45, n_samples).clip(20, 300)
chronic_conditions = np.random.poisson(lam=1, size=n_samples).clip(0, 4)

In [5]:
score_chronic = 0.08 * age + 0.5 * refill_frequency + 0.8 * chronic_conditions - 5.0
score_wellness = -0.04 * age + 0.02 * monthly_spend - 0.3 * chronic_conditions + 1.0
score_acute = np.zeros(n_samples)

In [6]:
exp_scores = np.exp(np.column_stack([score_acute, score_chronic, score_wellness]))
probs = exp_scores / exp_scores.sum(axis=1, keepdims=True)
target = np.array([np.random.choice([0, 1, 2], p=p) for p in probs])

In [7]:
df_pharmacy = pd.DataFrame({
    'age': age.round(1),
    'refill_frequency': refill_frequency,
    'monthly_spend': monthly_spend.round(2),
    'chronic_conditions': chronic_conditions,
    'category_target': target
})

In [8]:
category_map = {0: 'Acute Care', 1: 'Chronic Care', 2: 'Wellness/OTC'}
df_pharmacy['category_name'] = df_pharmacy['category_target'].map(category_map)

print("Target Class Distribution:")
print(df_pharmacy['category_name'].value_counts())

Target Class Distribution:
category_name
Chronic Care    189
Wellness/OTC    166
Acute Care       45
Name: count, dtype: int64


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [10]:
X = df_pharmacy[['age', 'refill_frequency', 'monthly_spend', 'chronic_conditions']]
y = df_pharmacy['category_target']

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

In [12]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [13]:
baseline_model = LogisticRegression(
    multi_class='multinomial',
    solver='lbfgs',
    penalty='l2',
    C=1.0,
    max_iter=1000,
    random_state=42
)

In [14]:
baseline_model.fit(X_train_scaled, y_train)
print("Baseline model successfully trained across classes:", baseline_model.classes_)

c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Baseline model successfully trained across classes: [0 1 2]


In [15]:
from sklearn.model_selection import GridSearchCV

In [16]:
param_grid = {
    'penalty': ['l1', 'l2'],
    'C': [0.01, 0.1, 1.0, 10.0],
    'solver': ['saga']  # 'saga' supports both L1 and L2 for multinomial loss
}

grid_search = GridSearchCV(
    estimator=LogisticRegression(multi_class='multinomial', max_iter=2000, random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)

In [17]:
grid_search.fit(X_train_scaled, y_train)
best_model = grid_search.best_estimator_

c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [18]:
print("--- Optimal Model Parameters ---")
print("Best Penalty:", grid_search.best_params_['penalty'])
print("Best Regularization Strength (C):", grid_search.best_params_['C'])
print("Best 5-Fold Cross-Validation F1-Macro Score:", round(grid_search.best_score_, 4))

--- Optimal Model Parameters ---
Best Penalty: l1
Best Regularization Strength (C): 10.0
Best 5-Fold Cross-Validation F1-Macro Score: 0.5319


In [19]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [20]:
y_pred = best_model.predict(X_test_scaled)
target_names = [category_map[i] for i in best_model.classes_]

In [21]:
print("--- Multi-Class Classification Performance ---")
print("Test Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

print("Confusion Matrix:")
cm = pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=[f'Actual {name}' for name in target_names],
    columns=[f'Pred {name}' for name in target_names]
)
print(cm)

--- Multi-Class Classification Performance ---
Test Accuracy: 0.66

Detailed Classification Report:
              precision    recall  f1-score   support

  Acute Care       0.00      0.00      0.00        11
Chronic Care       0.65      0.79      0.71        47
Wellness/OTC       0.71      0.69      0.70        42

    accuracy                           0.66       100
   macro avg       0.45      0.49      0.47       100
weighted avg       0.60      0.66      0.63       100

Confusion Matrix:
                     Pred Acute Care  Pred Chronic Care  Pred Wellness/OTC
Actual Acute Care                  0                  7                  4
Actual Chronic Care                2                 37                  8
Actual Wellness/OTC                0                 13                 29


In [22]:
coef_matrix = best_model.coef_
feature_names = X.columns.tolist()

In [23]:
odds_ratios = np.exp(coef_matrix)

In [24]:
df_coefs = pd.DataFrame(
    coef_matrix.T,
    index=feature_names,
    columns=[f'Coef ({category_map[i]})' for i in best_model.classes_]
)

In [25]:
df_odds = pd.DataFrame(
    odds_ratios.T,
    index=feature_names,
    columns=[f'Odds Ratio ({category_map[i]})' for i in best_model.classes_]
)

In [26]:
print("--- Log-Odds Coefficients ---")
print(df_coefs.round(3))

print("\n--- Feature Odds Ratios (Exp(Beta)) ---")
print(df_odds.round(3))

--- Log-Odds Coefficients ---
                    Coef (Acute Care)  Coef (Chronic Care)  \
age                             0.000                1.126   
refill_frequency                0.000                1.112   
monthly_spend                  -0.121                0.000   
chronic_conditions              0.000                0.607   

                    Coef (Wellness/OTC)  
age                              -0.536  
refill_frequency                 -0.317  
monthly_spend                     0.936  
chronic_conditions               -0.560  

--- Feature Odds Ratios (Exp(Beta)) ---
                    Odds Ratio (Acute Care)  Odds Ratio (Chronic Care)  \
age                                   1.000                      3.083   
refill_frequency                      1.000                      3.041   
monthly_spend                         0.886                      1.000   
chronic_conditions                    1.000                      1.835   

                    Odds Ratio (Welln